# Credit Analytics Risk – Gold Layer

## Zweck

Die Gold-Schicht bildet die fachlich aufbereitete und analysefertige Ebene
der Medallion Architecture.

Als Grundlage dient die technisch bereinigte Silver-Tabelle
`Data_Science.credit_risk_silver`.

In der Gold-Schicht werden die Daten für nachfolgende Analysen strukturiert
und vereinheitlicht.

Die Gold-Schicht enthält noch keine Spezialisierung auf ein konkretes
PD- oder LGD-Modell. Diese fachliche Spezialisierung erfolgt in einem
nachgelagerten Schritt.

## Input

`Data_Science.credit_risk_silver`

## Output

`Data_Science.credit_risk_gold`

## Verarbeitung

- Silver-Daten laden
- fachlich relevante Attribute strukturieren
- Kategorien und Werte standardisieren
- abgeleitete Analyseattribute erstellen
- abschließende Datenqualitätsprüfung
- Gold-Tabelle speichern

## Medallion Architecture

```text
dataset.csv
    │
    ▼
BRONZE
credit_risk_bronze
    │
    ▼
SILVER
credit_risk_silver
    │
    ▼
GOLD
credit_risk_gold


```

##### Tabelle Silver laden

In [0]:
# Silver-Tabelle laden

df_gold = spark.table("Data_Science.credit_risk_silver")

display(df_gold.limit(1))

id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag
1077501,5000,5000,4975,36 months,10.65,162.87,B,B2,null,10+ years,RENT,24000,Verified,2011-12-01,Fully Paid,n,https://lendingclub.com/browse/loanDetail.action?loan_id=1077501,Borrower added on 12/22/11 > I need to upgrade my business technologies.,credit_card,Computer,860xx,AZ,27.65,0.0,1985-01-01,735.0,739.0,1.0,3.0,0.0,13648.0,83.7,9.0,f,0.0,0.0,5863.1551866952,5833.84,5000.0,863.16,0.0,0.0,0.0,2015-01-01,171.62,2018-12-01,749.0,745.0,0.0,1.0,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N


# 2. Auswahl von relevanten Attributen

In [0]:
gold_cols = [
    "id",
    "loan_amnt",
    "funded_amnt",
    "funded_amnt_inv",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "issue_d",
    "loan_status",
    "purpose",
    "zip_code",
    "addr_state",
    "dti",
    "delinq_2yrs",
    "earliest_cr_line",
    "fico_range_low",
    "fico_range_high",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "last_credit_pull_d",
    "last_fico_range_high",
    "last_fico_range_low",
    "collections_12_mths_ex_med",
    "policy_code",
    "application_type",
    "acc_now_delinq",
    "chargeoff_within_12_mths",
    "delinq_amnt",
    "pub_rec_bankruptcies",
    "tax_liens",
    "hardship_flag",
    "disbursement_method",
    "debt_settlement_flag"
]

df_gold = df_gold.select(gold_cols)

display(df_gold.limit(1))

id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag
1077501,5000,5000,4975,36 months,10.65,162.87,B,B2,10+ years,RENT,24000,Verified,2011-12-01,Fully Paid,credit_card,860xx,AZ,27.65,0.0,1985-01-01,735.0,739.0,1.0,3.0,0.0,13648.0,83.7,9.0,0.0,0.0,5863.1551866952,5833.84,5000.0,863.16,0.0,0.0,0.0,2015-01-01,171.62,2018-12-01,749.0,745.0,0.0,1.0,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N


In [0]:
print("Anzahl Zeilen:", df_gold.count())
print("Anzahl Spalten:", len(df_gold.columns))

Anzahl Zeilen: 39717
Anzahl Spalten: 54


## 3. Standardisierung von kategorischen Attributen

In [0]:
from pyspark.sql.functions import col, trim, upper, when

categorical_cols = [
    "term",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "verification_status",
    "loan_status",
    "purpose",
    "addr_state",
    "initial_list_status",
    "application_type",
    "hardship_flag",
    "disbursement_method",
    "debt_settlement_flag"
]

for c in categorical_cols:
    if c in df_gold.columns:
        df_gold = df_gold.withColumn(
            c,
            trim(col(c))
        )

#### Groß-/Kleinschreibung vereinheitlichen

In [0]:
upper_cols = [
    "grade",
    "sub_grade",
    "home_ownership",
    "verification_status",
    "loan_status",
    "purpose",
    "addr_state",
    "initial_list_status",
    "application_type",
    "hardship_flag",
    "disbursement_method",
    "debt_settlement_flag"
]

for c in upper_cols:
    if c in df_gold.columns:
        df_gold = df_gold.withColumn(
            c,
            upper(col(c))
        )

**Kontrolle**

In [0]:
for c in categorical_cols:
    if c in df_gold.columns:
        print(f"\n--- {c} ---")
        df_gold.select(c).distinct().orderBy(c).show(100, truncate=False)

### Sandardisierung der Laufzeit

In [0]:
from pyspark.sql.functions import regexp_replace, trim, col

df_gold = df_gold.withColumn(
    "term",
    regexp_replace(trim(col("term")), "months", "").cast("integer")
)

## Check

In [0]:
display(
    df_gold.select("term").distinct().orderBy("term")
)

term
36
60


#### Beschäftigungsdauer standardisieren

In [0]:
from pyspark.sql.functions import regexp_replace, trim, col

df_gold = df_gold.withColumn(
    "emp_length",
    trim(col("emp_length"))
)

df_gold = df_gold.withColumn(
    "emp_length",
    regexp_replace(col("emp_length"), r"\+ years", "")
)

df_gold = df_gold.withColumn(
    "emp_length",
    regexp_replace(col("emp_length"), r" years", "")
)

df_gold = df_gold.withColumn(
    "emp_length",
    regexp_replace(col("emp_length"), r" year", "")
)

df_gold = df_gold.withColumn(
    "emp_length",
    regexp_replace(col("emp_length"), r"< 1", "0")
)

df_gold = df_gold.withColumn(
    "emp_length",
    col("emp_length").cast("integer")
)

#### Check

In [0]:
display(
    df_gold.select("emp_length")
    .distinct()
    .orderBy("emp_length")
)

emp_length
null
0
1
2
3
4
5
6
7
8


#### Gold-Daten prüfen

In [0]:
display(
    df_gold.select(
        "term",
        "emp_length",
        "grade",
        "sub_grade",
        "home_ownership",
        "verification_status",
        "loan_status"
    ).limit(1)
)

term,emp_length,grade,sub_grade,home_ownership,verification_status,loan_status
36 months,10+ years,B,B2,RENT,VERIFIED,FULLY PAID


### Datentypen kontrollieren

In [0]:
df_gold.select(
    "term",
    "emp_length",
    "grade",
    "sub_grade",
    "home_ownership",
    "verification_status",
    "loan_status"
).printSchema()

root
 |-- term: integer (nullable = true)
 |-- emp_length: integer (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- loan_status: string (nullable = true)



## 4. Business-KPIs

In diesem Abschnitt werden aus den aufbereiteten Gold-Daten zentrale
Kennzahlen für die geschäftliche Analyse des Kreditportfolios berechnet.

Die KPIs dienen der Beschreibung des Kreditportfolios und der Unterstützung
von Business- und Credit-Risk-Analysen.

```text

Portfolio
├── Anzahl Kredite
├── Kreditvolumen
└── durchschnittlicher Kreditbetrag

Kreditnehmer
├── durchschnittliches Einkommen
└── durchschnittlicher DTI

Kreditrisiko
├── Verteilung Loan Status
├── Verteilung Grade
└── durchschnittlicher FICO

Zahlungsverhalten
├── Total Payment
├── Recoveries
└── Recovery Rate

```


### 5.1 Portfolio-KPIs

Zur geschäftlichen Bewertung des Kreditportfolios werden zunächst zentrale
Portfolio-Kennzahlen berechnet.

Dazu gehören:

- Anzahl der Kredite
- Gesamtes Kreditvolumen
- durchschnittlicher Kreditbetrag
- durchschnittlicher Zinssatz
- durchschnittliches Jahreseinkommen
- durchschnittlicher DTI

In [0]:
from pyspark.sql.functions import count, sum, avg, round

df_kpi_portfolio = df_gold.select(
    count("id").alias("anzahl_kredite"),
    sum("loan_amnt").alias("gesamtes_kreditvolumen"),
    round(avg("loan_amnt"), 2).alias("durchschnittlicher_kreditbetrag"),
    round(avg("int_rate"), 2).alias("durchschnittlicher_zinssatz"),
    round(avg("annual_inc"), 2).alias("durchschnittliches_jahreseinkommen"),
    round(avg("dti"), 2).alias("durchschnittlicher_dti")
)

display(df_kpi_portfolio)

anzahl_kredite,gesamtes_kreditvolumen,durchschnittlicher_kreditbetrag,durchschnittlicher_zinssatz,durchschnittliches_jahreseinkommen,durchschnittlicher_dti
39717,445602650,11219.44,12.02,68968.92,13.32


### 5.2 Risiko-KPIs nach Kreditstatus

In [0]:
from pyspark.sql.functions import count, sum, avg, round, col, lit

df_kpi_status = (
    df_gold
    .groupBy("loan_status")
    .agg(
        count("id").alias("anzahl_kredite"),
        sum("loan_amnt").alias("kreditvolumen"),
        round(avg("loan_amnt"), 2).alias("durchschnittlicher_kreditbetrag"),
        round(avg("int_rate"), 2).alias("durchschnittlicher_zinssatz"),
        round(avg("dti"), 2).alias("durchschnittlicher_dti")
    )
    .orderBy(col("kreditvolumen").desc())
)

display(df_kpi_status)

loan_status,anzahl_kredite,kreditvolumen,durchschnittlicher_kreditbetrag,durchschnittlicher_zinssatz,durchschnittlicher_dti
FULLY PAID,34075,377220250,11070.29,11.72,13.21
CHARGED OFF,5642,68382400,12120.24,13.83,14.01


###5.3 prozentualen Anteil jedes Kreditstatus am gesamten Portfolio

In [0]:
total_loans = df_gold.count()

df_kpi_status_share = (
    df_gold
    .groupBy("loan_status")
    .agg(
        count("id").alias("anzahl_kredite"),
        sum("loan_amnt").alias("kreditvolumen")
    )
    .withColumn(
        "anteil_kredite_prozent",
        round(col("anzahl_kredite") / lit(total_loans) * 100, 2)
    )
    .orderBy(col("anteil_kredite_prozent").desc())
)

display(df_kpi_status_share)

loan_status,anzahl_kredite,kreditvolumen,anteil_kredite_prozent
FULLY PAID,34075,377220250,85.79
CHARGED OFF,5642,68382400,14.21


###5.4 Risiko-KPIs nach Kreditrating

In [0]:
from pyspark.sql.functions import count, sum, avg, round, col

df_kpi_grade = (
    df_gold
    .groupBy("grade")
    .agg(
        count("id").alias("anzahl_kredite"),
        sum("loan_amnt").alias("kreditvolumen"),
        round(avg("loan_amnt"), 2).alias("durchschnittlicher_kreditbetrag"),
        round(avg("int_rate"), 2).alias("durchschnittlicher_zinssatz"),
        round(avg("dti"), 2).alias("durchschnittlicher_dti"),
        round(avg("fico_range_low"), 2).alias("durchschnittlicher_fico")
    )
    .orderBy("grade")
)

display(df_kpi_grade.limit(1))

grade,anzahl_kredite,kreditvolumen,durchschnittlicher_kreditbetrag,durchschnittlicher_zinssatz,durchschnittlicher_dti,durchschnittlicher_fico
A,10085,86982400,8624.93,7.34,12.06,750.39


### 5.5 Risiko-KPIs nach Kreditrating

Die Kredite wurden anhand des Kreditratings (`grade`) nach Risikoklassen analysiert.

Dabei ergibt sich folgende Einteilung:

1. **A/B:** tendenziell niedrigere Risikoklassen
2. **C/D:** mittlere Risikoklassen
3. **E/F/G:** höhere Risikoklassen

Für jedes Rating wurden folgende Kennzahlen betrachtet:

- Anzahl der Kredite
- Kreditvolumen
- durchschnittlicher Kreditbetrag
- durchschnittlicher Zinssatz
- durchschnittlicher DTI
- durchschnittlicher FICO-Score

Dadurch lässt sich untersuchen, wie sich Kreditvolumen, Zinssatz und
Bonitätsmerkmale zwischen den verschiedenen Risikoklassen unterscheiden.

### 5.6 Risk-Performance nach Rating und Kreditstatus

In [0]:
from pyspark.sql.functions import count, sum, avg, round, col

df_kpi_grade_status = (
    df_gold
    .groupBy("grade", "loan_status")
    .agg(
        count("id").alias("anzahl_kredite"),
        sum("loan_amnt").alias("kreditvolumen"),
        round(avg("int_rate"), 2).alias("durchschnittlicher_zinssatz"),
        round(avg("dti"), 2).alias("durchschnittlicher_dti")
    )
    .orderBy("grade", "loan_status")
)

display(df_kpi_grade_status.limit(1))

grade,loan_status,anzahl_kredite,kreditvolumen,durchschnittlicher_zinssatz,durchschnittlicher_dti
A,CHARGED OFF,602,4695550,7.6,13.35


#### Anteil der Kreditstatus je Rating

In [0]:
from pyspark.sql.functions import count, round, col

df_grade_status_share = (
    df_gold
    .groupBy("grade", "loan_status")
    .agg(
        count("id").alias("anzahl_kredite")
    )
)

grade_totals = (
    df_gold
    .groupBy("grade")
    .agg(
        count("id").alias("gesamt_kredite_grade")
    )
)

df_grade_status_share = (
    df_grade_status_share
    .join(grade_totals, on="grade", how="left")
    .withColumn(
        "anteil_status_prozent",
        round(
            col("anzahl_kredite") /
            col("gesamt_kredite_grade") * 100,
            2
        )
    )
    .orderBy("grade", col("anteil_status_prozent").desc())
)

display(df_grade_status_share.limit(1))

grade,loan_status,anzahl_kredite,gesamt_kredite_grade,anteil_status_prozent
A,FULLY PAID,9483,10085,94.03


###  Risk-Performance nach Rating und Kreditstatus

In diesem Schritt wird das Kreditrating (`grade`) mit dem tatsächlichen Kreditstatus (`loan_status`) kombiniert.

Dadurch wird untersucht, wie sich die verschiedenen Risikoklassen hinsichtlich ihrer Kreditperformance unterscheiden.

Betrachtet werden:

- Anzahl der Kredite je Rating und Kreditstatus
- Kreditvolumen je Rating und Kreditstatus
- durchschnittlicher Zinssatz
- durchschnittlicher DTI
- prozentualer Anteil der einzelnen Kreditstatus innerhalb eines Ratings

Diese Analyse ermöglicht einen detaillierteren Vergleich der Kreditperformance zwischen den verschiedenen Risikoklassen.

### 5.7 Default-Flag erstellen

Zunächst definieren wir problematische Kreditstatus als default_flag = 1.

In [0]:
from pyspark.sql.functions import when, col

df_gold = df_gold.withColumn(
    "default_flag",
    when(
        col("loan_status").isin(
            "CHARGED OFF",
            "DEFAULT"
        ),
        1
    ).otherwise(0)
)

display(
    df_gold.select(
        "loan_status",
        "default_flag"
    ).distinct().orderBy("loan_status")
)

loan_status,default_flag
CHARGED OFF,1
FULLY PAID,0


##### Prüfung der Verteilung

In [0]:
from pyspark.sql.functions import count, sum, round, col, lit

total_loans = df_gold.count()

df_kpi_default = (
    df_gold
    .groupBy("default_flag")
    .agg(
        count("id").alias("anzahl_kredite"),
        sum("loan_amnt").alias("kreditvolumen")
    )
    .withColumn(
        "anteil_prozent",
        round(
            col("anzahl_kredite") / lit(total_loans) * 100,
            2
        )
    )
    .orderBy("default_flag")
)

display(df_kpi_default)

default_flag,anzahl_kredite,kreditvolumen,anteil_prozent
0,34075,377220250,85.79
1,5642,68382400,14.21


### Default-Flag

Für die weitere Risikoanalyse wird aus dem Kreditstatus (`loan_status`) eine binäre Risikokennzahl (`default_flag`) abgeleitet.

Dabei gilt:

- `default_flag = 1`: Kreditstatus `CHARGED OFF` oder `DEFAULT`
- `default_flag = 0`: alle anderen Kreditstatus

Zusätzlich wird die Anzahl der Kredite und das Kreditvolumen je Risikoklasse betrachtet.

Die Kennzahl dient als klare Zielvariable für die weitere Risikoanalyse und kann später auch für die Modellierung der Probability of Default (PD) verwendet werden.

### 5.8 Default-Rate nach Kreditrating

In [0]:
from pyspark.sql.functions import count, sum, round, col

df_kpi_default_grade = (
    df_gold
    .groupBy("grade")
    .agg(
        count("id").alias("anzahl_kredite"),
        sum("default_flag").alias("anzahl_defaults"),
        sum("loan_amnt").alias("kreditvolumen")
    )
    .withColumn(
        "default_rate_prozent",
        round(
            col("anzahl_defaults") /
            col("anzahl_kredite") * 100,
            2
        )
    )
    .orderBy("grade")
)

display(df_kpi_default_grade.limit(1))

grade,anzahl_kredite,anzahl_defaults,kreditvolumen,default_rate_prozent
A,10085,602,86982400,5.97


###  Default-Rate nach Kreditrating

In diesem Schritt wird die Default-Rate für die einzelnen Kreditratings (`grade`) berechnet.

Die Kennzahl zeigt, welcher Anteil der Kredite innerhalb einer Ratingklasse als Default eingestuft wurde.

Betrachtet werden:

- Anzahl der Kredite
- Anzahl der Defaults
- Kreditvolumen
- Default-Rate in Prozent

Die Default-Rate ermöglicht einen direkten Vergleich der tatsächlichen Kreditperformance zwischen den verschiedenen Risikoklassen.

Eine steigende Default-Rate bei schlechteren Ratings würde darauf hindeuten, dass das Kreditrating die tatsächliche Ausfallwahrscheinlichkeit des Portfolios sinnvoll differenziert.

## 6. Validierung der Gold-Tabelle

In [0]:
# Anzahl Zeilen
print("Anzahl Zeilen:", df_gold.count())

# Anzahl Spalten
print("Anzahl Spalten:", len(df_gold.columns))

# Schema prüfen
df_gold.printSchema()

# Beispiel-Daten anzeigen
display(df_gold.limit(1))

Anzahl Zeilen: 39717
Anzahl Spalten: 55
root
 |-- id: integer (nullable = true)
 |-- loan_amnt: integer (nullable = true)
 |-- funded_amnt: integer (nullable = true)
 |-- funded_amnt_inv: integer (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: integer (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: date (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: double (nullable = true)
 |-- delinq_2yrs: double (nullable = true)
 |-- earliest_cr_line: date (nullable = true)
 |-- fico_range_low: double (nullable = true)
 |-- fico_range_high: double (nullab

id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag,default_flag
1077501,5000,5000,4975,36 months,10.65,162.87,B,B2,10+ years,RENT,24000,VERIFIED,2011-12-01,FULLY PAID,CREDIT_CARD,860xx,AZ,27.65,0.0,1985-01-01,735.0,739.0,1.0,3.0,0.0,13648.0,83.7,9.0,0.0,0.0,5863.1551866952,5833.84,5000.0,863.16,0.0,0.0,0.0,2015-01-01,171.62,2018-12-01,749.0,745.0,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0,N,CASH,N,0


#### Prüfung des default_flag

In [0]:
display(
    df_gold.groupBy("default_flag")
    .count()
    .orderBy("default_flag")
)

default_flag,count
0,34075
1,5642


#### Prüfung auf Duplikate

In [0]:
total_rows = df_gold.count()
distinct_rows = df_gold.distinct().count()

print("Gesamtzahl Zeilen:", total_rows)
print("Eindeutige Zeilen:", distinct_rows)
print("Duplikate:", total_rows - distinct_rows)

Gesamtzahl Zeilen: 39717
Eindeutige Zeilen: 39717
Duplikate: 0


###  Validierung

Vor dem Speichern der Gold-Tabelle wird die finale Datenstruktur überprüft.

Dabei werden folgende Punkte kontrolliert:

- Anzahl der Zeilen und Spalten
- Datentypen und Schema
- Verteilung des `default_flag`
- mögliche Duplikate
- Plausibilität der finalen Daten

Die Validierung stellt sicher, dass die Gold-Tabelle für Business-Analysen und das spätere Reporting verwendet werden kann.

## 7. Gold-Tabelle speichern

In [0]:
gold_table = "Data_Science.credit_risk_gold"

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print(f"Gold-Tabelle erfolgreich gespeichert: {gold_table}")

Gold-Tabelle erfolgreich gespeichert: Data_Science.credit_risk_gold


## 8.  KPIs-Tabellen speichern

In [0]:
# Portfolio-KPIs
df_kpi_portfolio.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Data_Science.credit_risk_kpi_portfolio")

# KPIs nach Kreditstatus
df_kpi_status_share.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Data_Science.credit_risk_kpi_status")

# KPIs nach Rating
df_kpi_grade.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Data_Science.credit_risk_kpi_grade")

# Default-Rate nach Rating
df_kpi_default_grade.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Data_Science.credit_risk_kpi_default_grade")

## Abschluss


Der Gold Layer ist damit vollständig aufgebaut.

Die bereinigten und standardisierten Kreditdaten wurden um relevante
Business-Features und Risiko-Kennzahlen erweitert. Zusätzlich wurden
aggregierte KPIs für das Reporting erstellt.

### Projektstruktur

#### Medallion Architecture

```text
dataset.csv
    │
    ▼
BRONZE
credit_risk_bronze
    │
    ▼
SILVER
credit_risk_silver
    │
    ▼
GOLD
credit_risk_gold
    │
    ├── KPI Portfolio
    ├── KPI Loan Status
    ├── KPI Grade
    ├── KPI Default Rate
   